In [11]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Statsmodels for interpretable logistic regression output
import statsmodels.api as sm

# Scikit-learn
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, log_loss, precision_recall_curve
)

In [69]:
df = pd.read_csv("../data/data_preprocees_rf.csv")
df

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,target_is_fraud
0,34,82,38635.01,544.0,20,60.92,80.16,4.9,1,0,...,2025,11,4,101,0.228916,0.018916,0.000000,0.024390,0.338043,0
1,48,2,19912.97,703.0,21,112.11,571.12,0.3,0,0,...,2025,6,5,254,7.000000,0.067519,0.000000,0.000000,-2.474067,0
2,27,0,20326.87,720.0,25,73.61,492.57,4.6,1,0,...,2025,7,5,226,24.000000,0.043430,0.000000,0.000000,-1.356393,0
3,45,49,38452.47,703.0,17,47.53,204.18,25.3,1,0,...,2024,7,1,566,0.340000,0.014828,0.000000,0.028571,0.338043,0
4,37,46,31943.14,594.0,13,99.95,734.09,12.8,0,1,...,2025,5,2,271,0.276596,0.037534,0.013699,0.000000,2.005874,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,56,0,34775.62,727.0,21,51.72,226.11,3.8,0,0,...,2025,7,2,222,21.000000,0.017841,0.000000,0.000000,-2.474067,0
159996,41,4,88617.57,770.0,18,52.54,171.07,15.1,1,0,...,2024,7,5,576,3.600000,0.007114,0.000000,0.000000,1.239936,0
159997,30,2,41148.54,738.0,20,29.34,119.81,0.7,0,0,...,2024,8,4,549,6.666667,0.008554,0.000000,0.000000,-2.474067,0
159998,56,6,31943.14,719.0,25,88.56,553.16,22.6,2,0,...,2025,3,2,341,3.571429,0.033257,0.000000,0.000000,-0.238719,0


In [13]:
from statsmodels.stats import descriptivestats

df.describe()

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,target_is_fraud
count,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,...,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000
mean,38.180700,17.747238,36660.592187,670.094206,22.010288,60.783112,292.509249,11.898687,0.786775,0.048763,...,2024.498575,6.494819,2.989081,413.047650,3.123338,0.025967,0.000555,0.007991,-0.013781,0.030775
std,11.440212,17.020847,20308.522522,57.395624,4.622836,39.147629,237.971011,11.452111,0.858241,0.215372,...,0.500000,3.440732,2.001009,211.038192,4.124815,0.022956,0.002484,0.013583,2.186616,0.172708
min,18.000000,0.000000,8840.598500,531.000000,12.000000,5.460000,17.959800,0.100000,0.000000,0.000000,...,2024.000000,1.000000,0.000000,55.000000,0.228916,0.001470,0.000000,0.000000,-2.474067,0.000000
25%,30.000000,5.000000,22769.945000,632.000000,19.000000,32.320000,122.220000,3.500000,0.000000,0.000000,...,2024.000000,4.000000,1.000000,230.000000,0.818182,0.010429,0.000000,0.000000,-1.356393,0.000000
50%,38.000000,12.000000,31943.140000,670.000000,22.000000,52.540000,224.730000,8.300000,1.000000,0.000000,...,2024.000000,7.000000,3.000000,413.000000,1.600000,0.019268,0.000000,0.000000,-0.238719,0.000000
75%,46.000000,25.000000,44988.015000,708.000000,25.000000,79.820000,389.300000,16.700000,1.000000,0.000000,...,2025.000000,9.000000,5.000000,597.000000,3.500000,0.033548,0.000000,0.018868,1.239936,0.000000
max,66.000000,82.000000,115836.969200,809.000000,34.000000,195.270200,1188.503000,55.400000,3.000000,1.000000,...,2025.000000,12.000000,6.000000,770.000000,24.000000,0.124763,0.013699,0.057143,6.799901,1.000000


In [14]:
df_numeric = df.select_dtypes(exclude=['object', 'category'])

df_numeric = df_numeric.dropna(axis=1)

In [15]:
X = df_numeric.drop(columns="target_is_fraud")
y = df_numeric["target_is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [ ]:
pipe_rf = Pipeline([
    ("scaler", RobustScaler()),         # scaling, optionnel pour RF mais ok
    ("smote", SMOTE(random_state=42)), # équilibre les classes
    ("rf", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",       
        random_state=42,
        n_jobs=-1
    ))
])

pipe_rf.fit(X_train, y_train)

# Prédictions
y_proba_rf = pipe_rf.predict_proba(X_test)[:, 1]
y_pred_rf = pipe_rf.predict(X_test)

print("=== Random Forest - Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# Optimisation du seuil pour F1 fraude
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = np.argmax(f1_scores)
best_f1 = f1_scores[best_idx]
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0

print(f"\nSeuil optimal RF: {best_threshold:.3f} | F1 optimal fraude: {best_f1:.4f}")

=== Random Forest - Confusion Matrix ===
[[38769     0]
 [ 1214    17]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     38769
           1       1.00      0.01      0.03      1231

    accuracy                           0.97     40000
   macro avg       0.98      0.51      0.51     40000
weighted avg       0.97      0.97      0.96     40000


Seuil optimal RF: 0.145 | F1 optimal fraude: 0.0982


In [52]:
pipe_rf_opt = Pipeline([
    ("scaler", RobustScaler()),
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=100,
    min_samples_split=200,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
))
])

In [53]:
pipe_rf_opt.fit(X_train, y_train)


Pipeline(steps=[('scaler', RobustScaler()), ('smote', SMOTE(random_state=42)),
                ('rf',
                 RandomForestClassifier(class_weight='balanced', max_depth=8,
                                        min_samples_leaf=100,
                                        min_samples_split=200, n_estimators=400,
                                        n_jobs=-1, random_state=42))])

In [54]:
y_proba = pipe_rf_opt.predict_proba(X_test)[:, 1]

In [55]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
best_f1 = f1_scores[best_idx]

In [56]:
y_pred_opt = (y_proba >= best_threshold).astype(int)

In [57]:
print("=== Random Forest Optimisé ===")
print(f"Seuil optimal: {best_threshold:.3f}")
print(f"F1 optimal fraude: {best_f1:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_opt))

=== Random Forest Optimisé ===
Seuil optimal: 0.386
F1 optimal fraude: 0.0796

Confusion Matrix:
[[30778  7991]
 [  849   382]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87     38769
           1       0.05      0.31      0.08      1231

    accuracy                           0.78     40000
   macro avg       0.51      0.55      0.48     40000
weighted avg       0.94      0.78      0.85     40000



In [23]:
df_test = pd.read_csv("../data/data_preprocess_rf_test.csv")

In [24]:
df_test

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639.0,24,106.61,781.55,7.3,0,0,...,2025,9,1,140,0.444444,0.049974,0.0,0.000000,0.093756,CUST_E5RX1BC9II
1,22,5,45378.40,699.0,21,54.17,252.95,4.2,1,0,...,2024,7,2,573,3.500000,0.014321,0.0,0.000000,-1.350939,CUST_BHWIUKERGN
2,42,2,36643.70,765.0,17,44.40,238.79,7.0,1,0,...,2025,2,3,355,5.666667,0.014535,0.0,0.000000,-1.350939,CUST_EXT9NA4CHU
3,39,20,30283.30,573.0,29,38.30,321.44,28.3,0,0,...,2024,8,0,547,1.380952,0.015171,0.0,0.016949,-0.765594,CUST_9FSJE5R1NY
4,54,10,35294.22,624.0,21,70.93,540.48,21.7,0,0,...,2024,2,3,747,1.909091,0.024108,0.0,0.000000,-2.508966,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622.0,24,42.99,214.60,7.3,0,0,...,2025,12,3,75,6.000000,0.012418,0.0,0.020408,-0.765594,CUST_1OM9UCID91
39996,48,12,36783.80,697.0,25,21.17,69.19,3.8,1,0,...,2024,6,6,618,1.923077,0.006904,0.0,0.000000,-1.350939,CUST_VDEY72BIZP
39997,49,28,55963.60,658.0,18,180.58,410.17,3.6,2,0,...,2024,11,2,468,0.620690,0.038713,0.0,0.000000,-0.192911,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683.0,19,129.93,643.13,13.9,0,0,...,2024,6,5,612,2.375000,0.161561,0.0,0.000000,-2.508966,CUST_IXX0BE9VQD


In [25]:
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import LabelEncoder

# df_test = dataset brut de test avec les colonnes listées

df_test_processed = df_test.copy()

# 1️⃣ Features temporelles
if 'signup_date' in df_test_processed.columns:
    df_test_processed['signup_date'] = pd.to_datetime(df_test_processed['signup_date'], errors='coerce')
    df_test_processed['signup_year'] = df_test_processed['signup_date'].dt.year
    df_test_processed['signup_month'] = df_test_processed['signup_date'].dt.month
    df_test_processed['signup_day_of_week'] = df_test_processed['signup_date'].dt.dayofweek
    df_test_processed['days_since_signup'] = (datetime.now() - df_test_processed['signup_date']).dt.days
    df_test_processed.drop('signup_date', axis=1, inplace=True)

# 2️⃣ Features composites / ratios
if 'num_transactions_30d' in df_test_processed.columns and 'tenure_months' in df_test_processed.columns:
    df_test_processed['tx_per_month'] = df_test_processed['num_transactions_30d'] / (df_test_processed['tenure_months'] + 1)

if 'avg_amount_30d_eur' in df_test_processed.columns and 'annual_income_eur' in df_test_processed.columns:
    df_test_processed['tx_to_income_ratio'] = df_test_processed['avg_amount_30d_eur'] / (df_test_processed['annual_income_eur']/12 + 1)

if 'chargebacks_12m' in df_test_processed.columns and 'num_transactions_30d' in df_test_processed.columns:
    df_test_processed['chargeback_rate'] = df_test_processed['chargebacks_12m'] / (df_test_processed['num_transactions_30d'] * 4 + 1)

if 'failed_payments_6m' in df_test_processed.columns and 'num_transactions_30d' in df_test_processed.columns:
    df_test_processed['failed_payment_rate'] = df_test_processed['failed_payments_6m'] / (df_test_processed['num_transactions_30d'] * 2 + 1)

risk_cols = [c for c in ['chargebacks_12m', 'failed_payments_6m', 'support_tickets_90d', 'is_vpn', 'is_new_device'] 
             if c in df_test_processed.columns]
if risk_cols:
    risk_df = df_test_processed[risk_cols].copy()
    for col in risk_cols:
        if df_test_processed[col].std() > 0:
            risk_df[col] = (risk_df[col] - risk_df[col].mean()) / risk_df[col].std()
    df_test_processed['composite_risk_score'] = risk_df.sum(axis=1)

# 3️⃣ Encodage des catégorielles
categorical_cols = df_test_processed.select_dtypes(include=['object', 'category']).columns.tolist()

# Exclure customer_id
categorical_cols = [c for c in categorical_cols if c != 'customer_id']

label_encoders = {}
for col in categorical_cols:
    top_values = df_test_processed[col].value_counts().head(50).index
    df_test_processed[col] = df_test_processed[col].apply(lambda x: x if x in top_values else 'Other')
    le = LabelEncoder()
    df_test_processed[col] = le.fit_transform(df_test_processed[col].astype(str))
    label_encoders[col] = le  # en mémoire pour info

# 4️⃣ Traitement des outliers
numeric_cols = df_test_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
exclude_cols = ['customer_id', 'account_id', 'customer_note', 'referrer_code',
                'secondary_email', 'postal_code', 'city', 'last_ticket_subject',
                'post_event_status_code', 'chargeback_resolution_time_days',
                'manual_review_result', 'target_is_fraud']

features = [c for c in df_test_processed.columns if c not in exclude_cols]
X_test_new = df_test_processed[features].copy()

# Réinjecter customer_id pour l'export
if 'customer_id' in df_test_processed.columns:
    X_test_new['customer_id'] = df_test_processed['customer_id']

# Export CSV
X_test_new.to_csv('../data/data_preprocess_rf_test.csv', index=False)
print("✅ data_preprocess_rf_test.csv créé!")


# 6️⃣ Réinjecter customer_id pour l'export
if 'customer_id' in df_test_processed.columns:
    X_test_new['customer_id'] = df_test_processed['customer_id']

# 7️⃣ Export CSV
#X_test_new.to_csv('../data/data_preprocess_rf_test.csv', index=False)
print("✅ data_preprocess_rf_test.csv créé!")
print(f"Dataset final: {X_test_new.shape[0]} lignes × {X_test_new.shape[1]} colonnes")


✅ data_preprocess_rf_test.csv créé!
✅ data_preprocess_rf_test.csv créé!
Dataset final: 40000 lignes × 53 colonnes


In [26]:
X_test_new

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639.0,24,106.61,781.55,7.3,0,0,...,2025,9,1,140,0.444444,0.049974,0.0,0.000000,0.093756,CUST_E5RX1BC9II
1,22,5,45378.40,699.0,21,54.17,252.95,4.2,1,0,...,2024,7,2,573,3.500000,0.014321,0.0,0.000000,-1.350939,CUST_BHWIUKERGN
2,42,2,36643.70,765.0,17,44.40,238.79,7.0,1,0,...,2025,2,3,355,5.666667,0.014535,0.0,0.000000,-1.350939,CUST_EXT9NA4CHU
3,39,20,30283.30,573.0,29,38.30,321.44,28.3,0,0,...,2024,8,0,547,1.380952,0.015171,0.0,0.016949,-0.765594,CUST_9FSJE5R1NY
4,54,10,35294.22,624.0,21,70.93,540.48,21.7,0,0,...,2024,2,3,747,1.909091,0.024108,0.0,0.000000,-2.508966,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622.0,24,42.99,214.60,7.3,0,0,...,2025,12,3,75,6.000000,0.012418,0.0,0.020408,-0.765594,CUST_1OM9UCID91
39996,48,12,36783.80,697.0,25,21.17,69.19,3.8,1,0,...,2024,6,6,618,1.923077,0.006904,0.0,0.000000,-1.350939,CUST_VDEY72BIZP
39997,49,28,55963.60,658.0,18,180.58,410.17,3.6,2,0,...,2024,11,2,468,0.620690,0.038713,0.0,0.000000,-0.192911,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683.0,19,129.93,643.13,13.9,0,0,...,2024,6,5,612,2.375000,0.161561,0.0,0.000000,-2.508966,CUST_IXX0BE9VQD


In [60]:
features_model = [c for c in X_test_new.columns if c != 'customer_id']
X_features = X_test_new[features_model]

y_proba_new = pipe_rf_opt.predict_proba(X_features)[:, 1]

y_pred_new = (y_proba_new >= 0.40955352142949214).astype(int)

df_pred_final = pd.DataFrame({
    'customer_id': X_test_new['customer_id'],
    'target_is_fraud': y_pred_new
})

# 4️⃣ Export CSV
df_pred_final.to_csv('../Preds/RandomForest_preds_Noah_v2.csv', index=False)
print("✅ data_rf_predictions_final.csv créé !")
print(f"Dataset final: {df_pred_final.shape[0]} lignes × {df_pred_final.shape[1]} colonnes")


✅ data_rf_predictions_final.csv créé !
Dataset final: 40000 lignes × 2 colonnes


In [67]:
df_pred_final["target_is_fraud"].value_counts()

target_is_fraud
0    32973
1     7027
Name: count, dtype: int64

In [59]:
from sklearn.metrics import f1_score

y_train_pred = pipe_rf_opt.predict(X_train)
y_val_pred = pipe_rf_opt.predict(X_val)

print("F1 train :", f1_score(y_train, y_train_pred))
print("F1 valid :", f1_score(y_val, y_val_pred))


F1 train : 0.06872065055549192
F1 valid : 0.06050129645635264


In [ ]:
y_proba_val = pipe_rf_opt.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, y_proba_val)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)

best_idx = np.argmax(f1_scores)

print("Seuil optimal :", thresholds[best_idx])
print("F1 optimal :", f1_scores[best_idx])
print("Recall fraude :", recall[best_idx])
print("Precision fraude :", precision[best_idx])


Seuil optimal : 0.40955352142949214
F1 optimal : 0.08034057390265956
Recall fraude : 0.24915368991198375
Precision fraude : 0.04789172306090578
